In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# constant definitions unchanged
Brand31 = "Brand#31"
Brand43 = "Brand#43"
SMBOX = "SM BOX"
SMCASE = "SM CASE"
SMPACK = "SM PACK"
SMPKG = "SM PKG"
MEDBAG = "MED BAG"
MEDBOX = "MED BOX"
MEDPACK = "MED PACK"
MEDPKG = "MED PKG"
LGBOX = "LG BOX"
LGCASE = "LG CASE"
LGPACK = "LG PACK"
LGPKG = "LG PKG"
DELIVERINPERSON = "DELIVER IN PERSON"
AIR = "AIR"
AIRREG = "AIRREG"

# group container types into lists for fast membership checks
SM_SMALL = [SMBOX, SMCASE, SMPACK, SMPKG]
MED_CONTAINER = [MEDBAG, MEDBOX, MEDPACK, MEDPKG]
LG_CONTAINER = [LGBOX, LGCASE, LGPACK, LGPKG]
SHIPMODES = [AIR, AIRREG]

# 1) filter lineitem: ship instruction, ship mode, and quantity between 4 and 36
li_mask = (
    (lineitem.L_SHIPINSTRUCT == DELIVERINPERSON)
    & (lineitem.L_SHIPMODE.isin(SHIPMODES))
    & lineitem.L_QUANTITY.between(4, 36)
)
fline = lineitem.loc[
    li_mask,
    ["L_PARTKEY", "L_QUANTITY", "L_EXTENDEDPRICE", "L_DISCOUNT"]
]

# 2) filter part: brand, size, and container per the three groups
pt_mask = (
    (part.P_BRAND == Brand31) & part.P_SIZE.between(1, 5) & part.P_CONTAINER.isin(SM_SMALL)
    | (part.P_BRAND == Brand43) & part.P_SIZE.between(1, 10) & part.P_CONTAINER.isin(MED_CONTAINER)
    | (part.P_BRAND == Brand43) & part.P_SIZE.between(1, 15) & part.P_CONTAINER.isin(LG_CONTAINER)
)
fpart = part.loc[
    pt_mask,
    ["P_PARTKEY", "P_CONTAINER"]
]

# 3) merge the two filtered tables on part key
df = fline.merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")

# 4) apply the quantity ranges by container group
mask_qty = (
    df.P_CONTAINER.isin(SM_SMALL) & df.L_QUANTITY.between(4, 14)
    | df.P_CONTAINER.isin(MED_CONTAINER) & df.L_QUANTITY.between(15, 25)
    | df.P_CONTAINER.isin(LG_CONTAINER) & df.L_QUANTITY.between(26, 36)
)
df = df.loc[mask_qty]

# 5) compute the final total
total = (df.L_EXTENDEDPRICE * (1.0 - df.L_DISCOUNT)).sum()